In [10]:
from ultralytics import YOLO
import cv2
import numpy as np

In [11]:
model = YOLO("../notebooks/runs/detect/train/weights/best.pt")

In [12]:
cap = cv2.VideoCapture("../videos/cricket_ball_demo.mp4")

width = int(cap.get(3))
height = int(cap.get(4))
fps = int(cap.get(5))

In [13]:
out = cv2.VideoWriter(
    "../videos/parabolic_output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

In [14]:
positions = []

while True:

    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, conf=0.4)[0]

    if results.boxes is not None and len(results.boxes) > 0:

        x1, y1, x2, y2 = results.boxes.xyxy.cpu().numpy()[0]

        cx = int((x1+x2)/2)
        cy = int((y1+y2)/2)

        positions.append((cx,cy))

cap.release()


0: 640x384 (no detections), 58.0ms
Speed: 2.0ms preprocess, 58.0ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 48.3ms
Speed: 1.1ms preprocess, 48.3ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 45.6ms
Speed: 1.2ms preprocess, 45.6ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 44.6ms
Speed: 1.4ms preprocess, 44.6ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 41.2ms
Speed: 1.0ms preprocess, 41.2ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 45.9ms
Speed: 1.6ms preprocess, 45.9ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 48.9ms
Speed: 1.1ms preprocess, 48.9ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 44.4ms
Speed: 1.6ms preprocess, 44.4ms i

In [15]:
x = np.array([p[0] for p in positions])
y = np.array([p[1] for p in positions])

t = np.arange(len(positions))

coeff_x = np.polyfit(t, x, 2)
coeff_y = np.polyfit(t, y, 2)

poly_x = np.poly1d(coeff_x)
poly_y = np.poly1d(coeff_y)

t_smooth = np.linspace(0, len(positions)-1, 100)

x_curve = poly_x(t_smooth)
y_curve = poly_y(t_smooth)

In [16]:
cap = cv2.VideoCapture("../videos/cricket_ball_demo.mp4")

curve_points = [(int(x_curve[i]), int(y_curve[i])) for i in range(len(x_curve))]

In [17]:
while True:

    ret, frame = cap.read()
    if not ret:
        break

    for i in range(len(curve_points)-1):

        cv2.line(
            frame,
            curve_points[i],
            curve_points[i+1],
            (0,0,255),
            3
        )

    out.write(frame)

cap.release()
out.release()